# HAM10000 期末實驗 00：資料檢查（近似重複影像＋ISIC 2018 外部資料準備）

這份筆記本要**最先執行**，只需要 CPU。它做兩件事，對應報告參之三（一）與參之五 E4：

1. **HAM10000 近似重複檢查**：先用感知雜湊（pHash）找出「病灶編號不同、但影像相近」的候選，再用結構相似度（SSIM）確認是否為同一張照片。結果只回報，不改變資料切分。
2. **ISIC 2018 外部資料準備（E4）**：下載 ISIC 2018 Task 3 的驗證集（193 張）與測試集（1,512 張），與 HAM10000 比對影像編號、ISIC 封存庫病灶編號，以及「pHash 候選＋SSIM 確認」的同一照片檢查。

**E4 排除規則（正式版，事先決定）**：符合以下任一項才排除：影像編號相同、ISIC 封存庫病灶編號（IL_）相同、pHash 距離 ≤ 6 且 SSIM ≥ 0.90。只有 pHash 相近但 SSIM 未達門檻的影像不排除，另以「排除全部 pHash 候選」作為敏感度分析。

輸出都存在 Google 雲端硬碟的 `ham10000_project/data_check/`，Baseline 與 CNN 筆記本會讀取這裡的 `isic2018_external.csv`。

**使用方式**：執行階段 → 全部執行。第一次會要求授權 Google 雲端硬碟。約 20–40 分鐘（大部分是下載與計算雜湊）。
> **正式版**：以 Claude 原版與 ChatGPT 審查修正版合併。pHash 在皮膚鏡影像上誤報很多，因此只作為候選篩選，是否為同一張照片以 SSIM 確認。

## 0. 環境設定

In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'imagehash'], check=False)

USE_DRIVE = True          # 結果存到 Google 雲端硬碟，斷線也不會消失
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

BASE = os.environ.get('HAM_BASE') or ('/content/drive/MyDrive/ham10000_project' if (IN_COLAB and USE_DRIVE) else './ham10000_project')
CHECK_DIR = os.path.join(BASE, 'data_check')
ISIC_DIR = os.path.join(BASE, 'isic2018')
os.makedirs(CHECK_DIR, exist_ok=True); os.makedirs(ISIC_DIR, exist_ok=True)
print('專案資料夾：', BASE)

In [ ]:
import time, json, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed
import imagehash

SEEDS = [42, 43, 44, 45, 46]          # 與 Baseline、CNN 筆記本相同
VAL_RATIO, TEST_RATIO = 0.15, 0.15
CLASS_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# 近似重複的判定門檻：64 位元感知雜湊的漢明距離小於等於此值，列為「疑似重複」
# 這個值在比對前就固定，寫進報告，不依結果調整
PHASH_THRESHOLD = 6

# 同一張照片的確認門檻：兩張影像轉灰階、縮成 256×192 後的結構相似度（SSIM）
# 同一張照片經縮放或重新壓縮後通常仍很高，不同病灶即使 pHash 相同也會明顯較低
SSIM_SIZE = (256, 192)
SSIM_THRESHOLD = 0.90

## 1. 讀取 HAM10000

In [ ]:
DATA_DIR = os.environ.get('HAM_DATA_DIR')   # 已手動下載的話，改成資料夾路徑
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

meta_path, img_paths = None, {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f == 'HAM10000_metadata.csv' and meta_path is None:
            meta_path = os.path.join(root, f)
        if f.lower().endswith('.jpg'):
            img_paths.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
assert meta_path is not None, '找不到 HAM10000_metadata.csv'
df = pd.read_csv(meta_path)
assert (len(df) == 10015 or os.environ.get('HAM_SMOKE') == '1') and df.image_id.is_unique, 'HAM10000 影像數或影像 ID 不符合預期'
assert df.lesion_id.notna().all(), 'HAM10000 有病灶編號缺失，需先查證'
assert (df.groupby('lesion_id').dx.nunique() == 1).all(), '同一病灶編號對應不同診斷'
df['path'] = df['image_id'].map(img_paths)
assert df['path'].notna().all(), '有影像找不到檔案'
df = df.sort_values('image_id').reset_index(drop=True)
print(f'HAM10000：影像 {len(df):,} 張，病灶 {df.lesion_id.nunique():,} 個')

## 2. 計算感知雜湊

每張影像轉灰階、縮小後做離散餘弦轉換，得到 64 位元的雜湊值。兩張影像的雜湊值有幾個位元不同（漢明距離），就代表它們有多不像：0 表示幾乎一樣。結果會快取，重跑時直接讀取。

In [ ]:
def phash_int(path):
    return int(str(imagehash.phash(Image.open(path).convert('RGB'))), 16)

def hash_table(paths, ids, cache_file):
    if os.path.exists(cache_file):
        c = pd.read_csv(cache_file, dtype={'phash': str})
        if c['image_id'].tolist() == list(ids):
            return np.array([int(h, 16) for h in c['phash']], dtype=np.uint64)
    t0 = time.time()
    hs = Parallel(n_jobs=-1)(delayed(phash_int)(p) for p in paths)
    pd.DataFrame({'image_id': list(ids), 'phash': [f'{h:016x}' for h in hs]}).to_csv(cache_file, index=False)
    print(f'雜湊計算完成，{len(hs):,} 張，耗時 {time.time()-t0:.0f} 秒')
    return np.array(hs, dtype=np.uint64)

ham_hash = hash_table(df['path'], df['image_id'], os.path.join(CHECK_DIR, 'ham_phash.csv'))

_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    # a: (n,) uint64，b: (m,) uint64 → (n, m) 漢明距離
    x = (a[:, None] ^ b[None, :]).view(np.uint8).reshape(len(a), len(b), 8)
    return _POP[x].sum(-1, dtype=np.uint8)

## 3. HAM10000 內部的近似重複

In [ ]:
lesion = df['lesion_id'].values
pairs = []
CHUNK = 500
for s in range(0, len(df), CHUNK):
    d = hamming(ham_hash[s:s+CHUNK], ham_hash)
    ii, jj = np.nonzero(d <= PHASH_THRESHOLD)
    ii = ii + s
    keep = ii < jj                                   # 每組只記一次
    for i, j in zip(ii[keep], jj[keep]):
        pairs.append((i, j, int(d[i - s, j])))
pairs = pd.DataFrame(pairs, columns=['i', 'j', 'distance'])
pairs['same_lesion'] = lesion[pairs['i']] == lesion[pairs['j']]

n_same_lesion_imgs = (df.groupby('lesion_id').size() > 1).sum()
print(f'門檻 ≤ {PHASH_THRESHOLD}：共 {len(pairs):,} 組相近影像')
print(f'  同一病灶編號：{pairs.same_lesion.sum():,} 組（參考：有 {n_same_lesion_imgs:,} 個病灶擁有多張影像）')
print(f'  不同病灶編號（疑似重複）：{(~pairs.same_lesion).sum():,} 組')

In [ ]:
# 疑似重複清單：加上各種子依病灶切分時，兩張是否分在不同子集
from sklearn.model_selection import train_test_split
holdout = VAL_RATIO + TEST_RATIO
test_share = TEST_RATIO / holdout
def split_by_lesion(df, seed):
    les = df.groupby('lesion_id')['dx'].first()
    tr_l, tmp_l = train_test_split(les.index.values, test_size=holdout, stratify=les.values, random_state=seed)
    va_l, te_l = train_test_split(tmp_l, test_size=test_share, stratify=les.loc[tmp_l].values, random_state=seed)
    lid = df['lesion_id'].values
    return (np.where(np.isin(lid, tr_l))[0], np.where(np.isin(lid, va_l))[0], np.where(np.isin(lid, te_l))[0])

cand = pairs[~pairs.same_lesion].sort_values('distance').reset_index(drop=True)
near = pd.DataFrame({
    'image_a': df.image_id.values[cand.i], 'image_b': df.image_id.values[cand.j],
    'lesion_a': lesion[cand.i], 'lesion_b': lesion[cand.j],
    'dx_a': df.dx.values[cand.i], 'dx_b': df.dx.values[cand.j],
    'distance': cand.distance.values,
})
for seed in SEEDS:
    tr, va, te = split_by_lesion(df, seed)
    sub = np.empty(len(df), dtype=object); sub[tr], sub[va], sub[te] = 'train', 'val', 'test'
    a, b = sub[cand.i], sub[cand.j]
    near[f'seed{seed}_train_test'] = ((a == 'train') & (b == 'test')) | ((a == 'test') & (b == 'train'))
near['人工確認（是／否）'] = ''
near.to_csv(os.path.join(CHECK_DIR, 'ham_near_duplicates_review.csv'), index=False, encoding='utf-8-sig')

summary = {'門檻': PHASH_THRESHOLD, '疑似重複組數': len(near),
           '涉及影像數': len(set(near.image_a) | set(near.image_b)),
           '不同診斷類別的組數': int((near.dx_a != near.dx_b).sum())}
for seed in SEEDS:
    summary[f'種子 {seed} 跨訓練集與測試集的組數'] = int(near[f'seed{seed}_train_test'].sum())
pd.Series(summary).to_csv(os.path.join(CHECK_DIR, 'ham_near_duplicates_summary.csv'), header=['值'], encoding='utf-8-sig')
print(pd.Series(summary).to_string())
print('⚠ 感知雜湊小於等於 6 只代表疑似近似影像，不代表已確認同一病灶。')
print('⚠ 若人工確認有不同 lesion_id 的同病灶影像跨 train/test，')
print('  不得再稱病灶切分已完全排除洩漏，需在正式訓練前決定修正方案。')

In [ ]:
# 圖：距離最小的疑似重複組合，給組員人工確認（左右兩張為一組）
show = near.head(20)
if len(show):
    n = len(show); cols = 4
    rows_ = int(np.ceil(n / (cols // 2)))
    fig, axes = plt.subplots(rows_, cols, figsize=(3 * cols, 3 * rows_))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis('off')
    for k, r in show.reset_index(drop=True).iterrows():
        for s, side in enumerate(['a', 'b']):
            ax = axes[k // 2, (k % 2) * 2 + s]
            ax.imshow(Image.open(img_paths[r[f'image_{side}']]).convert('RGB'))
            ax.set_title(f"#{k+1} {r[f'image_{side}']}\n{r[f'lesion_{side}']} ({r[f'dx_{side}']}) d={r['distance']}", fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(CHECK_DIR, 'fig_ham_near_duplicates.png'), dpi=150)
    plt.show()
else:
    print('沒有疑似重複的影像')

### 3-1. 以 SSIM 確認是否為同一張照片

注意：下面以病灶編號是否相同來量化 pHash，反映的是「找出同病灶影像」的能力，不等於「找出同一張照片」的準確率。SSIM 0.90 是事先設定的篩選門檻，達門檻的組合仍需人工確認。

pHash 只比較整張圖的大致明暗分布。皮膚鏡影像大多是「膚色背景、中央一塊深色」，所以不同病灶也常被判為相近。下面先量化 pHash 的表現，再對每一組候選計算 SSIM。

In [ ]:
from skimage.metrics import structural_similarity

def gray_small(path):
    return np.asarray(Image.open(path).convert('L').resize(SSIM_SIZE, Image.BILINEAR), dtype=np.float32) / 255.0

def ssim_pairs(paths_a, paths_b):
    def one(a, b):
        return float(structural_similarity(gray_small(a), gray_small(b), data_range=1.0))
    return np.array(Parallel(n_jobs=-1)(delayed(one)(a, b) for a, b in zip(paths_a, paths_b)), dtype=float)

# pHash 對「同一病灶的不同照片」抓得到多少：同病灶影像組數 = Σ C(k, 2)
k = df.groupby('lesion_id').size()
same_lesion_pairs_total = int((k * (k - 1) // 2).sum())
print(f'同病灶的影像組合共 {same_lesion_pairs_total:,} 組，pHash ≤ {PHASH_THRESHOLD} 只找到 {int(pairs.same_lesion.sum()):,} 組'
      f'（{pairs.same_lesion.sum() / same_lesion_pairs_total:.1%}）')
print(f'pHash ≤ {PHASH_THRESHOLD} 的候選中，不同病灶編號占 {(~pairs.same_lesion).mean():.1%}')

near['ssim'] = ssim_pairs(df.set_index('image_id').loc[near.image_a, 'path'].values,
                          df.set_index('image_id').loc[near.image_b, 'path'].values)
near['ssim_confirmed'] = near['ssim'] >= SSIM_THRESHOLD
near.to_csv(os.path.join(CHECK_DIR, 'ham_near_duplicates_review.csv'), index=False, encoding='utf-8-sig')
# SSIM 達門檻的組合需人工看圖確認。確認為同一病灶者，寫進 01、02 的 LESION_MERGE，切分前合併為同一病灶
confirmed = near[near['ssim_confirmed']][['image_a', 'image_b', 'lesion_a', 'lesion_b', 'dx_a', 'dx_b', 'distance', 'ssim']]
confirmed.to_csv(os.path.join(CHECK_DIR, 'ham_confirmed_same_photo.csv'), index=False, encoding='utf-8-sig')
print('SSIM 達門檻、需人工確認的組合：'); print(confirmed.to_string(index=False))

summary['同病灶影像組合總數'] = same_lesion_pairs_total
summary['pHash 找到的同病灶組合'] = int(pairs.same_lesion.sum())
summary[f'SSIM ≥ {SSIM_THRESHOLD} 確認為同一照片的組數'] = int(near['ssim_confirmed'].sum())
for seed in SEEDS:
    summary[f'種子 {seed} 確認組中跨訓練集與測試集的組數'] = int((near['ssim_confirmed'] & near[f'seed{seed}_train_test']).sum())
pd.Series(summary).to_csv(os.path.join(CHECK_DIR, 'ham_near_duplicates_summary.csv'), header=['值'], encoding='utf-8-sig')
print(pd.Series(summary).to_string())
print('\nSSIM 分布（不同病灶編號的 pHash 候選）：')
print(near['ssim'].describe(percentiles=[.5, .9, .99]).round(3).to_string())

In [ ]:
# 圖：SSIM 最高的候選，給組員人工確認
show = near.sort_values('ssim', ascending=False).head(10).reset_index(drop=True)
if len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 2.6 * len(show)))
    axes = np.atleast_2d(axes)
    for k_, r in show.iterrows():
        for s_, side in enumerate(['a', 'b']):
            ax = axes[k_, s_]; ax.axis('off')
            ax.imshow(Image.open(img_paths[r[f'image_{side}']]).convert('RGB'))
            ax.set_title(f"{r[f'image_{side}']} {r[f'lesion_{side}']} ({r[f'dx_{side}']})\npHash d={r['distance']}  SSIM={r['ssim']:.3f}", fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(CHECK_DIR, 'fig_ham_near_duplicates_top_ssim.png'), dpi=150); plt.show()

**人工確認方式**：打開 `ham_near_duplicates_review.csv`，對照上圖（或自行開啟影像），在「人工確認」欄填「是」或「否」。人工確認只作為補充紀錄。報告回報：pHash 候選幾組、SSIM 確認為同一照片幾組，以及確認組中各種子跨越訓練集與測試集的組數。

## 4. 下載 ISIC 2018 Task 3 驗證集與測試集

來源：ISIC Challenge 官方資料頁（https://challenge.isic-archive.com/data/），授權 CC BY-NC。若自動下載失敗，請手動下載下面四個檔案，放到雲端硬碟的 `ham10000_project/isic2018/` 再重跑。

In [ ]:
URL = 'https://isic-archive.s3.amazonaws.com/challenges/2018/'
FILES = {
    'val_img': 'ISIC2018_Task3_Validation_Input.zip',
    'val_gt':  'ISIC2018_Task3_Validation_GroundTruth.zip',
    'test_img': 'ISIC2018_Task3_Test_Input.zip',
    'test_gt':  'ISIC2018_Task3_Test_GroundTruth.zip',
}
for key, name in FILES.items():
    zpath = os.path.join(ISIC_DIR, name)
    done_flag = zpath + '.extracted'
    if os.path.exists(done_flag):
        continue
    if not os.path.exists(zpath):
        print('下載', name)
        urllib.request.urlretrieve(URL + name, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(ISIC_DIR)
    open(done_flag, 'w').write('ok')
print('ISIC 2018 檔案就緒')

In [ ]:
gt_map = {'MEL': 'mel', 'NV': 'nv', 'BCC': 'bcc', 'AKIEC': 'akiec', 'BKL': 'bkl', 'DF': 'df', 'VASC': 'vasc'}
gts, isic_paths = [], {}
for root, _, files in os.walk(ISIC_DIR):
    for f in files:
        full = os.path.join(root, f)
        if f.endswith('.csv') and 'GroundTruth' in f and 'Task3' in f:
            g = pd.read_csv(full)
            g = g.rename(columns={g.columns[0]: 'image_id'})
            assert set(gt_map).issubset(g.columns), f'GT 缺少七類欄位：{f}'
            assert np.isclose(g[list(gt_map)].sum(axis=1), 1).all(), f'GT 不是七類單一標籤：{f}'
            g['dx'] = g[list(gt_map)].idxmax(axis=1).map(gt_map)
            g['subset'] = 'val' if 'Validation' in f else 'test'
            gts.append(g[['image_id', 'dx', 'subset']])
        if f.lower().endswith('.jpg'):
            isic_paths[os.path.splitext(f)[0]] = os.path.relpath(full, BASE)   # 存相對路徑，其他筆記本自行接上 BASE
ext = pd.concat(gts, ignore_index=True).sort_values(['subset', 'image_id']).reset_index(drop=True)
ext['path'] = ext['image_id'].map(isic_paths)
assert ext.image_id.is_unique, 'ISIC2018 有重複的影像 ID，請先確認是否重複讀到 GT'
assert ext['path'].notna().all(), '有 ISIC 影像找不到檔案'
print(ext.groupby('subset').size().rename('影像數').to_string())

## 5. 取得 ISIC 影像的病灶編號與提供機構

ISIC 封存庫為每張影像記錄了病灶編號（IL_ 開頭）與提供機構。HAM10000 影像在封存庫也有同樣的資訊，所以能比對 ISIC 2018 的影像是否和 HAM10000 屬於同一個病灶。
依序嘗試官方命令列工具 isic-cli 與網頁 API。兩者都失敗時，會跳過病灶編號比對並提示，報告中需說明此限制。

In [ ]:
META_FILE = os.path.join(ISIC_DIR, 'isic_archive_metadata.csv')
COLLECTIONS = {66: 'HAM10000（2018 訓練集）', 73: 'ISIC 2018 驗證集', 67: 'ISIC 2018 測試集'}

def find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    return None

def meta_via_cli():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'isic-cli'], check=False)
    frames = []
    for cid in COLLECTIONS:
        out = os.path.join(ISIC_DIR, f'_meta_{cid}.csv')
        r = subprocess.run(['isic', 'metadata', 'download', '--collections', str(cid), '-o', out], capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(out):
            r = subprocess.run(['isic', 'metadata', 'download', '--collections', str(cid)], capture_output=True, text=True)
            if r.returncode != 0 or not r.stdout.startswith('isic_id'):
                raise RuntimeError(r.stderr[-300:])
            open(out, 'w').write(r.stdout)
        m = pd.read_csv(out)
        frames.append(pd.DataFrame({'image_id': m['isic_id'], 'collection': cid,
                                    'lesion_id': m['lesion_id'] if 'lesion_id' in m else None,
                                    'attribution': m['attribution'] if 'attribution' in m else None}))
    return pd.concat(frames, ignore_index=True)

def meta_via_api():
    import requests
    frames = []
    for cid in COLLECTIONS:
        url = 'https://api.isic-archive.com/api/v2/images/search/'
        params = {'collections': str(cid), 'limit': 100}
        rows = []
        while url:
            r = requests.get(url, params=params, timeout=60); r.raise_for_status()
            js = r.json()
            for it in js['results']:
                rows.append({'image_id': it['isic_id'], 'collection': cid,
                             'lesion_id': find_key(it.get('metadata', {}), 'lesion_id'),
                             'attribution': it.get('attribution')})
            url, params = js.get('next'), None
        frames.append(pd.DataFrame(rows))
    return pd.concat(frames, ignore_index=True)

if os.path.exists(META_FILE):
    meta = pd.read_csv(META_FILE)
else:
    meta = None
    for name, fn in [('isic-cli', meta_via_cli), ('網頁 API', meta_via_api)]:
        try:
            meta = fn(); print(f'已透過 {name} 取得封存庫資料')
            break
        except Exception as e:
            print(f'{name} 失敗：{str(e)[:200]}')
    if meta is not None:
        meta.to_csv(META_FILE, index=False)
if meta is None:
    print('⚠ 無法取得病灶編號與提供機構，將只用影像編號與感知雜湊比對，報告需說明此限制')
else:
    print(meta.groupby('collection').agg(影像數=('image_id', 'size'), 有病灶編號=('lesion_id', lambda s: s.notna().sum())).rename(index=COLLECTIONS).to_string())

## 6. 比對重疊並決定排除

In [ ]:
ext_hash = hash_table([os.path.join(BASE, p) for p in ext['path']], ext['image_id'], os.path.join(CHECK_DIR, 'isic2018_phash.csv'))
min_d = np.concatenate([hamming(ext_hash[s:s+CHUNK], ham_hash).min(1) for s in range(0, len(ext), CHUNK)])
ext['phash_min_distance'] = min_d
ext['overlap_image_id'] = ext['image_id'].isin(set(df['image_id']))
print('ISIC 影像與最相近 HAM10000 影像的漢明距離分布：')
print(pd.Series(min_d).describe(percentiles=[.01, .05, .25, .5]).round(1).to_string())
ext['overlap_phash'] = ext['phash_min_distance'] <= PHASH_THRESHOLD

# pHash 候選逐一以 SSIM 與「所有距離 ≤ 門檻的 HAM 影像」比對，取最高者
ham_paths = df['path'].values
cand_rows = []
for s in range(0, len(ext), CHUNK):
    d = hamming(ext_hash[s:s+CHUNK], ham_hash)
    ii, jj = np.nonzero(d <= PHASH_THRESHOLD)
    for i, j in zip(ii + s, jj):
        cand_rows.append((i, j, int(d[i - s, j])))
cand = pd.DataFrame(cand_rows, columns=['ext_idx', 'ham_idx', 'distance'])
cand['ssim'] = ssim_pairs([os.path.join(BASE, p) for p in ext['path'].values[cand.ext_idx]], ham_paths[cand.ham_idx]) if len(cand) else []
cand['isic_image_id'] = ext['image_id'].values[cand.ext_idx]
cand['ham_image_id'] = df['image_id'].values[cand.ham_idx]
cand.drop(columns=['ext_idx', 'ham_idx']).to_csv(os.path.join(CHECK_DIR, 'isic2018_phash_candidates_ssim.csv'), index=False, encoding='utf-8-sig')
best = cand.sort_values('ssim', ascending=False).drop_duplicates('ext_idx').set_index('ext_idx')
ext['max_ssim'] = pd.Series(best['ssim']).reindex(range(len(ext))).values
ext['max_ssim_ham_id'] = pd.Series(best['ham_image_id']).reindex(range(len(ext))).values
ext['overlap_same_photo'] = ext['max_ssim'] >= SSIM_THRESHOLD
print(f'pHash 候選 {int(ext.overlap_phash.sum())} 張，與候選 HAM 影像共比對 {len(cand)} 組')
print('候選的最高 SSIM 分布：')
print(ext.loc[ext.overlap_phash, 'max_ssim'].describe(percentiles=[.5, .9]).round(3).to_string())

if meta is not None:
    info = meta.drop_duplicates('image_id').set_index('image_id')
    ext['lesion_id'] = ext['image_id'].map(info['lesion_id'])
    ext['attribution'] = ext['image_id'].map(info['attribution'])
    ham_ids = set(df['image_id'])
    ham_lesions = set(info.loc[info.index.isin(ham_ids), 'lesion_id'].dropna())
    ham_sources = set(info.loc[info.index.isin(ham_ids), 'attribution'].dropna())
    ext['overlap_lesion'] = ext['lesion_id'].isin(ham_lesions)
    ext['source'] = np.where(ext['attribution'].isna(), 'unknown',
                             np.where(ext['attribution'].isin(ham_sources), 'same', 'different'))
    print('\n提供機構（attribution）分布：')
    print(meta.groupby('collection')['attribution'].value_counts().rename('影像數').to_string())
else:
    ext['lesion_id'], ext['attribution'] = None, None
    ext['overlap_lesion'] = False
    ext['source'] = 'unknown'

# 正式排除：影像編號相同、IL 病灶編號相同，或確認為同一張照片
ext['exclude'] = ext['overlap_image_id'] | ext['overlap_lesion'] | ext['overlap_same_photo']
ext['exclude_reason'] = np.select(
    [ext['overlap_image_id'], ext['overlap_lesion'], ext['overlap_same_photo']],
    ['same_image_id', 'same_lesion_id', 'same_photo_ssim'], default='kept')
# 敏感度分析：連同所有 pHash 候選一起排除
ext['exclude_sensitivity'] = ext['exclude'] | ext['overlap_phash']

# 病灶編號比對是否有效：HAM 的封存庫 IL 編號必須和 Kaggle 的 HAM 病灶編號一對一
il_valid = False
if meta is not None:
    il = df[['image_id', 'lesion_id']].merge(info['lesion_id'].rename('IL'), left_on='image_id', right_index=True, how='left')
    il_valid = bool(il['IL'].notna().all()
                    and (il.groupby('lesion_id')['IL'].nunique() == 1).all()
                    and (il.groupby('IL')['lesion_id'].nunique() == 1).all())
    print('\nHAM 病灶編號與封存庫 IL 編號一對一：', il_valid)

metadata_ok = False
coverage = {'ham_metadata_coverage': 0, 'external_metadata_coverage': 0,
            'ham_lesion_coverage': 0, 'external_lesion_coverage': 0}
if meta is not None and 'lesion_id' in meta.columns and 'image_id' in meta.columns:
    mm = meta.drop_duplicates('image_id').set_index('image_id')
    coverage = {
        'ham_metadata_coverage': int(df.image_id.isin(mm.index).sum()),
        'external_metadata_coverage': int(ext.image_id.isin(mm.index).sum()),
        'ham_lesion_coverage': int(df.image_id.map(mm.lesion_id).notna().sum()),
        'external_lesion_coverage': int(ext.image_id.map(mm.lesion_id).notna().sum()),
    }
    metadata_ok = (coverage['ham_metadata_coverage'] == len(df) and
                   coverage['external_metadata_coverage'] == len(ext) and
                   coverage['ham_lesion_coverage'] == len(df) and
                   coverage['external_lesion_coverage'] == len(ext))

audit = {'e4_ready_for_provisional_evaluation': bool(metadata_ok and il_valid),
         'e4_status': ('archive_lesion_metadata_complete_and_il_mapping_valid' if (metadata_ok and il_valid)
                       else 'INCOMPLETE_METADATA_OR_IL_MAPPING__REVIEW_BEFORE_E4'),
         'exclusion_rule': 'same image_id OR same archive lesion_id (IL) OR (pHash <= %d AND SSIM >= %.2f)' % (PHASH_THRESHOLD, SSIM_THRESHOLD),
         'sensitivity_rule': 'main rule OR pHash <= %d' % PHASH_THRESHOLD,
         'phash_threshold': int(PHASH_THRESHOLD), 'ssim_threshold': SSIM_THRESHOLD,
         'il_mapping_one_to_one': il_valid,
         'e4_image_count': int(len(ext)),
         'e4_kept_count_main': int((~ext['exclude']).sum()),
         'e4_kept_count_sensitivity': int((~ext['exclude_sensitivity']).sum()),
         'counts': coverage,
         'note': ('Lesion-level overlap is checked with ISIC archive lesion IDs. '
                  'Complete metadata does not prove the absence of every near-duplicate photo.')}
with open(os.path.join(CHECK_DIR, 'e4_audit_status.json'), 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
if not audit['e4_ready_for_provisional_evaluation']:
    print('⚠ 病灶 metadata 不完整或 IL 編號無法一對一，E4 分析會暫停。訓練可以照常進行。')
    print('  metadata 覆蓋：', coverage)
ext.to_csv(os.path.join(CHECK_DIR, 'isic2018_external.csv'), index=False, encoding='utf-8-sig')

overlap = pd.DataFrame({
    '影像數': ext.groupby('subset').size(),
    '影像編號重疊': ext.groupby('subset')['overlap_image_id'].sum(),
    'IL 病灶編號重疊': ext.groupby('subset')['overlap_lesion'].sum(),
    'pHash 候選': ext.groupby('subset')['overlap_phash'].sum(),
    'SSIM 確認同一照片': ext.groupby('subset')['overlap_same_photo'].sum(),
    '正式排除': ext.groupby('subset')['exclude'].sum(),
    '正式保留': ext.groupby('subset')['exclude'].apply(lambda s: (~s).sum()),
    '敏感度分析保留': ext.groupby('subset')['exclude_sensitivity'].apply(lambda s: (~s).sum()),
})
overlap.loc['合計'] = overlap.sum()
overlap.to_csv(os.path.join(CHECK_DIR, 'isic2018_overlap_summary.csv'), encoding='utf-8-sig')
print(overlap.to_string())

dist = pd.DataFrame({'全部': ext['dx'].value_counts(),
                     '正式保留': ext.loc[~ext['exclude'], 'dx'].value_counts(),
                     '敏感度分析保留': ext.loc[~ext['exclude_sensitivity'], 'dx'].value_counts()}).reindex(CLASS_ORDER).fillna(0).astype(int)
dist.loc['合計'] = dist.sum()
dist.to_csv(os.path.join(CHECK_DIR, 'isic2018_kept_class_distribution.csv'), encoding='utf-8-sig')
print('\n類別分布：')
print(dist.to_string())
print('\n來源分布：')
print(pd.crosstab(ext['subset'], ext['source'], margins=True).to_string())

## 7. 輸出檔案對照報告

| 檔案 | 用途 |
|---|---|
| `ham_near_duplicates_summary.csv` | 參之三（一）近似重複檢查結果、研究限制（含 pHash 的漏抓與誤報量化） |
| `ham_near_duplicates_review.csv`、`fig_ham_near_duplicates_top_ssim.png` | 組員人工確認用 |
| `isic2018_external.csv` | **Baseline 與 CNN 筆記本 E4 會讀取**，含正式排除與敏感度分析兩種標記 |
| `isic2018_phash_candidates_ssim.csv` | 每一組 pHash 候選的 SSIM，供查核 |
| `isic2018_overlap_summary.csv`、`isic2018_kept_class_distribution.csv` | 參之五 E4 的資料描述 |
| `e4_audit_status.json` | E4 排除規則與 metadata 稽核紀錄 |

「同來源（same）」表示該影像的提供機構也出現在 HAM10000，「不同來源（different）」較能代表新機構的資料。若來源顯示為 unknown，代表沒有取得封存庫資料，報告中 E4 只回報整體結果。